<a href="https://colab.research.google.com/github/ishancoderr/WiSee/blob/main/rssi_human_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# Load data from CSV file
def load_data(csv_file_path):
    # Read the CSV file
    df = pd.read_csv(csv_file_path)

    # Convert columns to numeric (in case they're loaded as strings)
    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    # Display data information
    print("Data overview:")
    print(df.head())
    print("\nClass distribution:")
    print(df['Human Status'].value_counts())

    return df

# Prepare data for modeling
def prepare_data(df):
    # Extract features and target
    X = df[['Receiver 1', 'Receiver 2', 'Receiver 3']].values
    y = df['Human Status'].values

    # Split data
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    # Scale features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Reshape for CNN (samples, timesteps, features)
    X_train_cnn = X_train_scaled.reshape(X_train_scaled.shape[0], X_train_scaled.shape[1], 1)
    X_test_cnn = X_test_scaled.reshape(X_test_scaled.shape[0], X_test_scaled.shape[1], 1)

    # One-hot encode targets
    y_train_categorical = to_categorical(y_train)
    y_test_categorical = to_categorical(y_test)

    return X_train_cnn, X_test_cnn, y_train_categorical, y_test_categorical, y_train, y_test

# Build CNN model
def create_cnn_model(num_classes):
    model = Sequential()

    # CNN layers
    model.add(Conv1D(filters=64, kernel_size=2, activation='relu', input_shape=(3, 1)))
    model.add(MaxPooling1D(pool_size=1))
    model.add(Conv1D(filters=32, kernel_size=1, activation='relu'))

    # Fully connected layers
    model.add(Flatten())
    model.add(Dense(64, activation='relu'))
    model.add(Dropout(0.5))
    model.add(Dense(num_classes, activation='softmax'))

    # Compile model
    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

# Train model with early stopping
def train_model(model, X_train, y_train, X_test, y_test, epochs=100, batch_size=32):
    # Define early stopping
    early_stopping = EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True,
        verbose=1
    )

    # Train the model
    history = model.fit(
        X_train, y_train,
        epochs=epochs,
        batch_size=batch_size,
        validation_data=(X_test, y_test),
        callbacks=[early_stopping],
        verbose=1
    )

    return model, history

# Evaluate model and visualize results
def evaluate_model(model, X_test, y_test_categorical, y_test, history):
    # Evaluate model
    test_loss, test_accuracy = model.evaluate(X_test, y_test_categorical)
    print(f"Test accuracy: {test_accuracy:.4f}")

    # Get predictions
    y_pred_proba = model.predict(X_test)
    y_pred = np.argmax(y_pred_proba, axis=1)

    # Classification report
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred,
                              target_names=['Empty (0)', 'Stationary (1)', 'Moving (2)']))

    # Confusion matrix
    plt.figure(figsize=(10, 7))
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Empty', 'Stationary', 'Moving'],
                yticklabels=['Empty', 'Stationary', 'Moving'])
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title('Confusion Matrix')
    plt.savefig('confusion_matrix.png')
    plt.close()

    # Training history
    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    plt.plot(history.history['accuracy'])
    plt.plot(history.history['val_accuracy'])
    plt.title('Model Accuracy')
    plt.ylabel('Accuracy')
    plt.xlabel('Epoch')
    plt.legend(['Train', 'Validation'], loc='upper left')

    plt.subplot(1, 2, 2)
    plt.plot(history.history['loss'])
    plt.plot(history.history['val_loss'])
    plt.title('Model Loss')
    plt.ylabel('Loss')
    plt.xlabel('Epoch')
    plt.legend(['Train', 'Validation'], loc='upper left')

    plt.tight_layout()
    plt.savefig('training_history.png')
    plt.close()

    return y_pred

# Function to make predictions on new data
def predict_human_status(model, new_data, scaler):
    # Scale new data
    new_data_scaled = scaler.transform(new_data)

    # Reshape for CNN
    new_data_cnn = new_data_scaled.reshape(new_data_scaled.shape[0], new_data_scaled.shape[1], 1)

    # Make predictions
    predictions = model.predict(new_data_cnn)
    predicted_classes = np.argmax(predictions, axis=1)

    # Map predictions to labels
    status_map = {0: 'Empty Room', 1: 'Stationary Person', 2: 'Moving Person'}
    predicted_labels = [status_map[cls] for cls in predicted_classes]

    return predicted_classes, predicted_labels, predictions

# Main execution function
def main(csv_file_path):
    # Load and prepare data
    df = load_data(csv_file_path)
    X_train, X_test, y_train_cat, y_test_cat, y_train, y_test = prepare_data(df)

    # Get number of classes
    num_classes = len(np.unique(y_train))
    print(f"Number of classes: {num_classes}")

    # Create and train model
    model = create_cnn_model(num_classes)
    model.summary()

    trained_model, history = train_model(model, X_train, y_train_cat, X_test, y_test_cat)

    # Evaluate model
    y_pred = evaluate_model(trained_model, X_test, y_test_cat, y_test, history)

    # Save the model
    trained_model.save('rssi_human_detection_model.h5')
    print("Model saved as 'rssi_human_detection_model.h5'")

    return trained_model

if __name__ == "__main__":
    # Specify the path to your CSV file
    csv_file_path = "https://raw.githubusercontent.com/ishancoderr/WiSee/refs/heads/main/data/finalDataset.csv"

    # Run the main function
    model = main(csv_file_path)

Data overview:
   Tile No  Receiver 1  Receiver 2  Receiver 3  Human Status
0        1         -37         -42         -34             0
1        1         -37         -42         -32             0
2        1         -37         -42         -32             0
3        1         -36         -41         -34             0
4        1         -39         -42         -35             0

Class distribution:
Human Status
2    818
1    734
0    629
Name: count, dtype: int64
Number of classes: 3


/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv1d_2 (Conv1D)                    │ (None, 2, 64)               │             192 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling1d_1 (MaxPooling1D)       │ (None, 2, 64)               │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv1d_3 (Conv1D)                    │ (None, 2, 32)               │           2,080 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten_1 (Flatten)                  │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 64)                  │           4,160 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_3 (Dense)                      │ (None, 3)                   │             195 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 6,627 (25.89 KB)

 Trainable params: 6,627 (25.89 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
55/55 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - accuracy: 0.6568 - loss: 0.9329 - val_accuracy: 0.8993 - val_loss: 0.4078
Epoch 2/100
55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8799 - loss: 0.3752 - val_accuracy: 0.9085 - val_loss: 0.2413
Epoch 3/100
55/55 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9056 - loss: 0.2477 - val_accuracy: 0.9268 - val_loss: 0.2190
Epoch 4/100
55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9089 - loss: 0.2406 - val_accuracy: 0.9108 - val_loss: 0.1981
Epoch 5/100
55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9252 - loss: 0.1962 - val_accuracy: 0.9153 - val_loss: 0.1818
Epoch 6/100
55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9237 - loss: 0.1876 - val_accuracy: 0.9176 - val_loss: 0.1793
Epoch 7/100
55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9270 - loss: 0.1827 - val_accuracy: 0.9382 - val_loss: 0.1617
Epoch 8/100
55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9449 - loss: 0.1608 - val_accuracy: 0.9314 - 

Model saved as 'rssi_human_detection_model.h5'


In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import seaborn as sns
from joblib import dump

# Load data from CSV file
def load_data(csv_file_path):
    # Read the CSV file
    df = pd.read_csv(csv_file_path)

    # Convert columns to numeric (in case they're loaded as strings)
    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    # Display data information
    print("Data overview:")
    print(df.head())
    print("\nClass distribution:")
    print(df['Human Status'].value_counts())

    return df

# Prepare data for modeling
def prepare_data(df):
    # Extract features and target
    X = df[['Receiver 1', 'Receiver 2', 'Receiver 3']].values
    y = df['Human Status'].values

    # Split data
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    # Scale features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    return X_train_scaled, X_test_scaled, y_train, y_test, scaler

# Function to train the Random Forest model with hyperparameter tuning
def train_random_forest(X_train, y_train):
    # Create parameter grid for GridSearchCV
    param_grid = {
        'n_estimators': [50, 100, 200],
        'max_depth': [None, 10, 20, 30],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4]
    }

    # Create base model
    rf = RandomForestClassifier(random_state=42)

    # Create GridSearchCV object
    print("Performing grid search for best hyperparameters...")
    grid_search = GridSearchCV(
        estimator=rf,
        param_grid=param_grid,
        cv=5,
        n_jobs=-1,
        verbose=1,
        scoring='accuracy'
    )

    # Fit the grid search
    grid_search.fit(X_train, y_train)

    # Get best model
    best_rf = grid_search.best_estimator_

    # Print best parameters
    print(f"Best parameters: {grid_search.best_params_}")
    print(f"Best cross-validation score: {grid_search.best_score_:.4f}")

    return best_rf

# Evaluate model and visualize results
def evaluate_model(model, X_test, y_test):
    # Make predictions
    y_pred = model.predict(X_test)

    # Calculate accuracy
    accuracy = accuracy_score(y_test, y_pred)
    print(f"Test accuracy: {accuracy:.4f}")

    # Classification report
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred,
                                target_names=['Empty (0)', 'Stationary (1)', 'Moving (2)']))

    # Confusion matrix
    plt.figure(figsize=(10, 7))
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Empty', 'Stationary', 'Moving'],
                yticklabels=['Empty', 'Stationary', 'Moving'])
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title('Confusion Matrix - Random Forest')
    plt.savefig('rf_confusion_matrix.png')
    plt.close()

    # Feature importance
    feature_importance = model.feature_importances_
    features = ['Receiver 1', 'Receiver 2', 'Receiver 3']

    plt.figure(figsize=(10, 6))
    sns.barplot(x=feature_importance, y=features)
    plt.title('Feature Importance - Random Forest')
    plt.xlabel('Importance')
    plt.ylabel('Feature')
    plt.tight_layout()
    plt.savefig('rf_feature_importance.png')
    plt.close()

    return y_pred

# Function to make predictions on new data
def predict_human_status(model, new_data, scaler):
    # Scale new data
    new_data_scaled = scaler.transform(new_data)

    # Make predictions
    predicted_classes = model.predict(new_data_scaled)
    predicted_proba = model.predict_proba(new_data_scaled)

    # Map predictions to labels
    status_map = {0: 'Empty Room', 1: 'Stationary Person', 2: 'Moving Person'}
    predicted_labels = [status_map[cls] for cls in predicted_classes]

    return predicted_classes, predicted_labels, predicted_proba

# Main execution function
def main(csv_file_path):
    # Load and prepare data
    df = load_data(csv_file_path)
    X_train, X_test, y_train, y_test, scaler = prepare_data(df)

    # Train Random Forest model
    print("Training Random Forest model...")
    rf_model = train_random_forest(X_train, y_train)

    # Evaluate model
    print("Evaluating model...")
    y_pred = evaluate_model(rf_model, X_test, y_test)

    # Save the model and scaler
    dump(rf_model, 'rssi_rf_model.joblib')
    dump(scaler, 'rssi_rf_scaler.joblib')
    print("Model saved as 'rssi_rf_model.joblib'")
    print("Scaler saved as 'rssi_rf_scaler.joblib'")

    # Example of making predictions with new data
    print("\nExample of making predictions with sample data:")
    sample_data = np.array([
        [-37, -42, -34],  # Example of empty room
        [-41, -43, -35],  # Example of stationary person
        [-38, -35, -37]   # Example of moving person
    ])

    predicted_classes, predicted_labels, _ = predict_human_status(rf_model, sample_data, scaler)

    for i, (data, pred_class, pred_label) in enumerate(zip(sample_data, predicted_classes, predicted_labels)):
        print(f"Sample {i+1}: RSSI values {data} -> Predicted class: {pred_class} ({pred_label})")

    return rf_model, scaler

if __name__ == "__main__":
    # Specify the path to your CSV file
    csv_file_path = "https://raw.githubusercontent.com/ishancoderr/WiSee/refs/heads/main/data/finalDataset.csv"

    # Run the main function
    rf_model, scaler = main(csv_file_path)

Data overview:
   Tile No  Receiver 1  Receiver 2  Receiver 3  Human Status
0        1         -37         -42         -34             0
1        1         -37         -42         -32             0
2        1         -37         -42         -32             0
3        1         -36         -41         -34             0
4        1         -39         -42         -35             0

Class distribution:
Human Status
2    818
1    734
0    629
Name: count, dtype: int64
Training Random Forest model...
Performing grid search for best hyperparameters...
Fitting 5 folds for each of 108 candidates, totalling 540 fits
Best parameters: {'max_depth': None, 'min_samples_leaf': 4, 'min_samples_split': 2, 'n_estimators': 200}
Best cross-validation score: 0.9730
Evaluating model...
Test accuracy: 0.9748

Classification Report:
                precision    recall  f1-score   support

     Empty (0)       0.98      0.98      0.98       126
Stationary (1)       0.96      0.98      0.97       147
    Moving